# Checkpoints

PathSim supports saving and loading simulation state via checkpoints. This allows you to pause a simulation, save its complete state to disk, and resume it later from exactly where you left off. 

Checkpoints also enable **rollback**, where you return to a saved state and explore different what-if scenarios by changing parameters.

Checkpoints use a split format: a JSON file for metadata and structure, and an NPZ file for numerical data (block states, solver histories, etc.).

## Setup

We'll simulate a driven harmonic oscillator — a mass-spring system excited by an external sinusoidal force. The system produces a sustained periodic response, making it easy to visually verify that checkpoints preserve continuity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathsim import Simulation, Connection
from pathsim.blocks import Integrator, Amplifier, Adder, Scope

In [ ]:
import numpy as np

# System parameters
m = 1.0    # mass
c = 0.1    # light damping
k = 4.0    # spring stiffness
F0 = 1.0   # forcing amplitude
w = 1.8    # forcing frequency (near resonance for k/m=4 -> w0=2)

def make_system(damping=c, stiffness=k):
    """Create a driven harmonic oscillator with configurable parameters."""
    from pathsim.blocks import Source, Integrator, Amplifier, Adder, Scope

    Src = Source(lambda t: F0/m * np.sin(w * t))  # external acceleration
    I1 = Integrator(0.0)       # velocity
    I2 = Integrator(0.5)       # position (start displaced)
    Ac = Amplifier(-damping/m)
    Ak = Amplifier(-stiffness/m)
    P1 = Adder()
    Sc = Scope(labels=["position"])

    blocks = [Src, I1, I2, Ac, Ak, P1, Sc]
    connections = [
        Connection(I1, I2, Ac),       # velocity -> position integrator, damper
        Connection(I2, Ak, Sc),       # position -> spring, scope
        Connection(Ac, P1),           # -c/m * v -> adder
        Connection(Ak, P1[1]),        # -k/m * x -> adder
        Connection(Src, P1[2]),       # F/m -> adder
        Connection(P1, I1),           # acceleration -> velocity integrator
    ]

    sim = Simulation(blocks, connections, dt=0.01)
    return sim, Sc

## Save Checkpoint

Run the simulation for 20 seconds, then save a checkpoint. The system will be in a sustained oscillation by this point.

In [ ]:
sim, scope = make_system()
sim.run(20)

# Save checkpoint
sim.save_checkpoint("checkpoint")
print(f"Saved checkpoint at t = {sim.time:.1f}s")

## Resume from Checkpoint

Load the checkpoint into a fresh simulation and continue for another 20 seconds. The new simulation has completely different Python objects, yet the checkpoint restores the exact state by matching blocks by type and insertion order.

In [ ]:
sim_resumed, scope_resumed = make_system()
sim_resumed.load_checkpoint("checkpoint")
print(f"Resumed from t = {sim_resumed.time:.1f}s")

sim_resumed.run(20)

## Rollback: What-If Scenarios

This is where checkpoints really shine. We reload the same checkpoint but with **different parameters** — increasing the damping significantly. Both branches start from the exact same state at t=20, but evolve differently.

In [ ]:
# Scenario A: same parameters (continuation)
sim_a, scope_a = make_system(damping=0.1)
sim_a.load_checkpoint("checkpoint")
sim_a.run(20)

# Scenario B: increased damping (what-if)
sim_b, scope_b = make_system(damping=1.5)
sim_b.load_checkpoint("checkpoint")
sim_b.run(20)

# Scenario C: stiffer spring (what-if)
sim_c, scope_c = make_system(stiffness=9.0)
sim_c.load_checkpoint("checkpoint")
sim_c.run(20)

## Compare Results

The plot shows the original simulation (0–20s), followed by three different futures branching from the same checkpoint.

In [ ]:
time_orig, data_orig = scope.read()
time_a, data_a = scope_a.read()
time_b, data_b = scope_b.read()
time_c, data_c = scope_c.read()

fig, ax = plt.subplots(figsize=(10, 4))

# Original run (0-20s)
ax.plot(time_orig, data_orig[0], "k-", lw=1.5, label="original (c=0.1, k=4)")

# Three futures from checkpoint
ax.plot(time_a, data_a[0], "C0-", alpha=0.8, label="resumed (c=0.1, k=4)")
ax.plot(time_b, data_b[0], "C1-", alpha=0.8, label="what-if: heavy damping (c=1.5)")
ax.plot(time_c, data_c[0], "C2-", alpha=0.8, label="what-if: stiffer spring (k=9)")

ax.axvline(20, color="gray", ls=":", alpha=0.5, lw=2, label="checkpoint (t=20s)")
ax.set_xlabel("time [s]")
ax.set_ylabel("position")
ax.set_title("Checkpoint Rollback: Three Futures from the Same State")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

All three scenarios start from the exact same state at t=20s. The blue continuation matches the original trajectory perfectly, while the heavy damping scenario (orange) decays rapidly and the stiffer spring scenario (green) shifts to a higher natural frequency.

## Checkpoint File Contents

The JSON file contains human-readable metadata about the simulation state. Let's inspect it.

In [ ]:
import json

with open("checkpoint.json") as f:
    cp = json.load(f)

print(f"PathSim version: {cp['pathsim_version']}")
print(f"Simulation time: {cp['simulation']['time']:.1f}s")
print(f"Solver: {cp['simulation']['solver']}")
print(f"Blocks saved: {len(cp['blocks'])}")
for b in cp["blocks"]:
    print(f"  {b['_key']} ({b['type']})")

Blocks are matched by type and insertion order (`Integrator_0`, `Integrator_1`, etc.), which means the checkpoint can be loaded into any simulation with the same block structure, regardless of the specific Python objects.